# Fish Audio Drama 3 (G4 / H100 / A100 · SGLang-Omni Zero-Shot Multi-Ref TTS & Novel Narration)

Serves `fishaudio/drama-3` (BF16 on G4 / H100 / A100-80GB) or `fishaudio/drama-3-int4` (auto-selected on <72 GiB GPUs or via `--variant int4`) through `sglang-omni[fishaudio]==0.2.0` (3-stage `preprocessing` -> `tts_engine` -> `vocoder`, 44.1 kHz mono output, ~22x aggregate realtime throughput on G4) with fail-closed Bearer token auth, multi-reference voice cloning, expressive inline event tags (`[sighs]`, `[laughs]`), `/v1/audio/speech/batch` novel narration, and persistent Google Drive voices/outputs.

Run it with the service script (`colabllm <name> up`), which creates the Colab session, mounts Drive with `colab dfs`, optionally creates a named Cloudflare tunnel, and injects `CT_*` settings. Cell order is the standard contract -- see `.agents/skills/colab-llm-service/SKILL.md`.

In [ ]:
# @title 0. Config -- values come from `colab exec --env` (defaults allow running in the Colab UI)
import json, os, re, secrets, shutil, stat, subprocess, sys, time, urllib.request, urllib.error

SERVICE      = os.environ.get("CT_SERVICE", "drama3")
SERVICE_PORT = int(os.environ.get("CT_SERVICE_PORT", "8000"))
AUTH         = os.environ.get("CT_AUTH", "builtin")            # caddy | builtin | none
PUBLIC_PORT  = int(os.environ.get("CT_PUBLIC_PORT", "8000"))  # what cloudflared targets
AUTH_USER    = os.environ.get("CT_AUTH_USER", "admin")
AUTH_PASS    = os.environ.get("CT_AUTH_PASS") or secrets.token_urlsafe(18)
API_TOKEN    = os.environ.get("CT_API_TOKEN") or "sk-" + secrets.token_urlsafe(32)
TUNNEL_TOKEN = os.environ.get("CT_TUNNEL_TOKEN", "")            # set -> fixed domain (named tunnel)
PUBLIC_HOST  = os.environ.get("CT_PUBLIC_HOSTNAME", "")
CREDS_FROM_ENV = bool(os.environ.get("CT_AUTH_PASS"))

WORK = "/content/ct"                                            # logs + binaries of this notebook
os.makedirs(WORK, exist_ok=True)
for _p in ("/content/drive/MyDrive/colab-llm/hf_token.txt",
           "/content/drive/MyDrive/colab-services/hf_token.txt",
           "/content/drive/MyDrive/colab-services/qwen-flash/hf_token.txt"):
    if not os.environ.get("HF_TOKEN") and os.path.isfile(_p):
        _tok = open(_p).read().strip()
        if _tok:
            os.environ["HF_TOKEN"] = os.environ["HUGGING_FACE_HUB_TOKEN"] = _tok
            break
MOUNT_DRIVE = os.environ.get("CT_MOUNT_DRIVE", "1") != "0"
DATA_DIR = os.environ.get("CT_DATA_DIR", f"/content/drive/MyDrive/colab-llm/{SERVICE}")
if not MOUNT_DRIVE or not os.path.isdir("/content/drive/MyDrive"):   # Drive not mounted -> local fallback
    if DATA_DIR.startswith("/content/drive/"):
        DATA_DIR = f"/content/data/{SERVICE}"
os.makedirs(DATA_DIR, exist_ok=True)
PROCS = {}                                                      # name -> (Popen, restart function)

def ct_spawn(name, args, env=None, cwd=None):
    """Start a background process; output goes to /content/ct/<name>.log."""
    with open(f"{WORK}/{name}.log", "a") as logf:   # child keeps its own dup of the fd
        return subprocess.Popen(args, stdout=logf, stderr=subprocess.STDOUT, cwd=cwd,
                                env={**os.environ, **(env or {})}, start_new_session=True)

def ct_stop_proc(proc, timeout=15):
    """Terminate a ct_spawn process group, escalating to SIGKILL after a bounded wait."""
    import signal
    if proc is None or proc.poll() is not None:
        return
    for sig in (signal.SIGTERM, signal.SIGKILL):
        try:
            os.killpg(proc.pid, sig)
        except (ProcessLookupError, PermissionError):
            try:
                proc.send_signal(sig)
            except ProcessLookupError:
                return
        try:
            proc.wait(timeout=timeout)
            return
        except subprocess.TimeoutExpired:
            continue

def ct_gpu_stat(query="memory.used,memory.total,utilization.gpu"):
    """nvidia-smi summary that never raises (CPU runtimes / transient driver errors)."""
    try:
        return subprocess.check_output(["nvidia-smi", f"--query-gpu={query}", "--format=csv,noheader"],
                                       text=True, stderr=subprocess.DEVNULL, timeout=20).strip()
    except (OSError, subprocess.SubprocessError):
        return "cpu/no-gpu"

def ct_log_size(name):
    p = f"{WORK}/{name}.log"
    return os.path.getsize(p) if os.path.exists(p) else 0

def ct_log_since(name, offset):
    p = f"{WORK}/{name}.log"
    if not os.path.exists(p):
        return ""
    with open(p, "rb") as handle:
        handle.seek(min(offset, os.path.getsize(p)))
        return handle.read().decode("utf-8", errors="replace")

def ct_public_gate(public_url, probe, verify_auth, log, timeout=120, interval=8):
    """Wait for the public endpoint; auth failures are fatal, network errors are retried."""
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        try:
            code = probe()
        except Exception as err:   # DNS / edge propagation / TLS warm-up
            log(f"Public probe pending ({type(err).__name__}: {err})")
            time.sleep(interval)
            continue
        if code == 200:
            verify_auth()            # RuntimeError propagates: never report CT_READY with broken auth
            return
        log(f"Public probe pending (HTTP {code})")
        time.sleep(interval)
    raise RuntimeError(f"Public endpoint {public_url} failed readiness verification within {timeout}s")

def ct_watchdog(health, start_service, log, *, service_key="service", log_name="service",
                interval=60, grace=900, dead_markers=(), heartbeat=None):
    """Supervise PROCS forever: restart dead processes, debounce health failures after reloads."""
    started = time.monotonic()
    failures = 0
    grace_until = time.monotonic()
    log_offset = ct_log_size(log_name)

    def restart_service(reason):
        nonlocal failures, grace_until, log_offset
        log(f"[watchdog] {reason}; restarting {service_key}")
        ct_stop_proc(PROCS[service_key][0])
        PROCS[service_key] = (start_service(), start_service)
        failures = 0
        grace_until = time.monotonic() + grace
        log_offset = ct_log_size(log_name)

    while True:
        time.sleep(interval)
        for name, (proc, restart) in list(PROCS.items()):
            if proc.poll() is not None:
                log(f"[watchdog] {name} exited with {proc.returncode}; restarting\n{ct_tail(name, 20)}")
                PROCS[name] = (restart(), restart)
                if name == service_key:
                    failures = 0
                    grace_until = time.monotonic() + grace
                    log_offset = ct_log_size(log_name)
        if dead_markers:
            fresh = ct_log_since(log_name, log_offset)
            log_offset += len(fresh.encode("utf-8", errors="replace"))
            if any(marker in fresh for marker in dead_markers):
                restart_service("Detected dead worker marker in service log")
                continue
        try:
            healthy = bool(health())
        except Exception:
            healthy = False
        if healthy:
            failures = 0
        elif time.monotonic() >= grace_until:
            failures += 1
        if failures >= 6:
            restart_service("6 consecutive failed health checks")
        if heartbeat is not None:
            try:
                heartbeat(int(time.monotonic() - started))
            except Exception as err:
                log(f"[watchdog] heartbeat warning: {err}")

def ct_wait_http(url, timeout=600, ok=lambda code: code < 500):
    """Wait until url answers with a non-5xx status (401/404 count as 'listening')."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                if ok(r.status): return r.status
        except urllib.error.HTTPError as e:
            if ok(e.code): return e.code
        except Exception:
            pass
        time.sleep(2)
    raise TimeoutError(f"{url} not ready after {timeout}s")

def ct_tail(name, n=30):
    p = f"{WORK}/{name}.log"
    return "".join(open(p, errors="replace").readlines()[-n:]) if os.path.exists(p) else ""

def ct_fetch_binary(path, url):
    if not os.path.exists(path):
        partial = f"{path}.{os.getpid()}.partial"
        with urllib.request.urlopen(url, timeout=60) as src, open(partial, "wb") as dst:
            shutil.copyfileobj(src, dst)
        os.chmod(partial, os.stat(partial).st_mode | stat.S_IEXEC)
        os.replace(partial, path)
    return path

print(f"service={SERVICE} port={SERVICE_PORT} auth={AUTH} public_port={PUBLIC_PORT} "
      f"tunnel={'named ' + PUBLIC_HOST if TUNNEL_TOKEN else 'quick'} data={DATA_DIR}")


In [ ]:
# @title 1. Setup -- install SGLang-Omni 0.2.0 + stage Fish Audio Drama 3 & S2-Pro codec on local SSD
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

VARIANT_REPOS = {
    "bf16": "fishaudio/drama-3",
    "drama-3": "fishaudio/drama-3",
    "int4": "fishaudio/drama-3-int4",
    "drama-3-int4": "fishaudio/drama-3-int4",
}
CODEC_REPO = "fishaudio/s2-pro"
CODEC_REVISION = "1de9996b6be38b745688de084d87a5633f714e4e"

def dlog(msg):
    print(time.strftime("[%F %T] [drama3] ") + msg, flush=True)

_gpu_raw = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv,noheader,nounits"],
    text=True,
).strip().splitlines()[0]
_gpu_parts = [p.strip() for p in _gpu_raw.split(",")]
GPU_NAME = _gpu_parts[0]
GPU_VRAM_MIB = int(float(_gpu_parts[1])) if len(_gpu_parts) > 1 else 0
GPU_ARCH = (_gpu_parts[2] if len(_gpu_parts) > 2 else "8.0").replace(".", "")

RAW_VARIANT = os.environ.get("CT_DRAMA3_VARIANT", "auto").strip().lower()
EXPLICIT_MODEL = os.environ.get("CT_DRAMA3_MODEL", "").strip()
if EXPLICIT_MODEL:
    MODEL_REPO = EXPLICIT_MODEL
    VARIANT = "int4" if "int4" in EXPLICIT_MODEL.lower() else "bf16"
elif RAW_VARIANT in VARIANT_REPOS:
    MODEL_REPO = VARIANT_REPOS[RAW_VARIANT]
    VARIANT = "int4" if "int4" in RAW_VARIANT else "bf16"
else:
    VARIANT = "bf16" if GPU_VRAM_MIB >= 72000 else "int4"
    MODEL_REPO = VARIANT_REPOS[VARIANT]

MODEL_ALIAS = os.environ.get("CT_DRAMA3_ALIAS", "drama3").strip() or "drama3"
ENGINE_GPU_MEM = float(os.environ.get("CT_DRAMA3_ENGINE_GPU_MEM", "0.70"))
VOCODER_GPU_MEM = float(os.environ.get("CT_DRAMA3_VOCODER_GPU_MEM", "0.10"))
MAX_RUNNING_REQS = int(os.environ.get("CT_DRAMA3_MAX_RUNNING_REQS", "16"))
CONTEXT_LENGTH = int(os.environ.get("CT_DRAMA3_CONTEXT_LENGTH", "16384"))
BATCH_MAX_ITEMS = int(os.environ.get("CT_DRAMA3_BATCH_MAX_ITEMS", "32"))
MAX_REF_SEC = float(os.environ.get("CT_DRAMA3_MAX_REF_SEC", "60.0"))
SGL_OMNI_VER = os.environ.get("CT_DRAMA3_SGL_OMNI_VERSION", "0.2.0").strip() or "0.2.0"

D3_WORK = Path("/content/drama3")
VENV_DIR = D3_WORK / "venv"
MODELS_ROOT = D3_WORK / "models"
MODEL_DIR = MODELS_ROOT / MODEL_REPO.split("/")[-1]
CODEC_DIR = D3_WORK / "codec"
SPEAKERS_DIR = D3_WORK / "speakers"
DRIVE_VOICES_DIR = Path(DATA_DIR) / "voices"
DRIVE_OUTPUT_DIR = Path(DATA_DIR) / "outputs"
DEFAULT_REF_WAV = D3_WORK / "default_ref.wav"
DEFAULT_REF_TEXT = (
    "Okay. Yeah. I resent you. I love you. I respect you. "
    "But you know what? You blew it! And thanks to you."
)

# Local NVMe SSD compiler caches archived as single tarballs on Google Drive
FLASHINFER_CACHE_DIR = D3_WORK / "flashinfer-cache"
TRITON_CACHE_DIR = D3_WORK / "triton-cache"
INDUCTOR_CACHE_DIR = D3_WORK / "inductor-cache"
FLASHINFER_LEGACY_DIR = Path(DATA_DIR) / "flashinfer-cache"
TRITON_LEGACY_DIR = Path(DATA_DIR) / "triton-cache"
INDUCTOR_LEGACY_DIR = Path(DATA_DIR) / "inductor-cache"

for _d in (
    D3_WORK, MODELS_ROOT, CODEC_DIR, SPEAKERS_DIR, DRIVE_VOICES_DIR, DRIVE_OUTPUT_DIR,
    FLASHINFER_CACHE_DIR, TRITON_CACHE_DIR, INDUCTOR_CACHE_DIR,
):
    _d.mkdir(parents=True, exist_ok=True)

(D3_WORK / "colab_cache.py").write_text('"""Stage optional compiler caches using one Drive archive per environment/GPU."""\nimport hashlib\nimport os\nimport shutil\nimport tarfile\nimport tempfile\nimport threading\nimport time\nimport uuid\nfrom pathlib import Path\n\n\ndef _progress(log, message, operation):\n    stopped = threading.Event()\n    started = time.monotonic()\n\n    def heartbeat():\n        while not stopped.wait(10):\n            log(f\'{message}, elapsed {time.monotonic()-started:.0f} s\')\n\n    log(message)\n    thread = threading.Thread(target=heartbeat, daemon=True)\n    thread.start()\n    try:\n        return operation()\n    finally:\n        stopped.set()\n        thread.join()\n\n\ndef _files(root):\n    return sorted(p for p in root.rglob(\'*\') if p.is_file() and not p.is_symlink()\n                  and not p.name.endswith((\'.lock\', \'.tmp\')))\n\n\ndef _fingerprint(root, files):\n    digest = hashlib.sha256()\n    for path in files:\n        stat = path.stat()\n        digest.update(f\'{path.relative_to(root)}:{stat.st_size}:{stat.st_mtime_ns}\\n\'.encode())\n    return digest.hexdigest()\n\n\ndef _unpack(archive, local):\n    # Validate the whole archive before extracting; compiler caches contain\n    # regular files/directories only. Never follow archive or local symlinks.\n    with tarfile.open(archive, \'r:\') as tar:\n        members = tar.getmembers()\n        root = local.resolve()\n        for member in members:\n            destination = local / member.name\n            if (not (member.isfile() or member.isdir()) or Path(member.name).is_absolute()\n                    or not destination.resolve().is_relative_to(root)):\n                raise ValueError(\'Cache archive contains unsafe paths or file types\')\n            current = destination\n            while current != local:\n                if current.is_symlink():\n                    raise ValueError(\'Cache target contains a symbolic link\')\n                current = current.parent\n        tar.extractall(local, members=members, filter=\'data\')\n\n\ndef restore_cache(local, legacy, archive_dir, key, log=print):\n    local, legacy, archive_dir = map(Path, (local, legacy, archive_dir))\n    local.mkdir(parents=True, exist_ok=True)\n    archive = archive_dir / key / (local.name + \'.tar\')\n    stamp = local.parent / (\'.\' + local.name + \'-\' + key + \'.restored\')\n    if archive.is_file():\n        stat = archive.stat()\n        signature = f\'{stat.st_size}:{stat.st_mtime_ns}\'\n        if stamp.exists() and stamp.read_text() == signature and _files(local):\n            log(f\'{local.name}: already restored locally, skipping duplicate transfer\')\n            return \'local\'\n        try:\n            with tempfile.TemporaryDirectory(dir=local.parent, prefix=\'.cache-restore-\') as tmp:\n                staged = Path(tmp) / \'cache.tar\'\n                _progress(log, f\'{local.name}: reading archive from Drive ({stat.st_size/1e6:.1f} MB)\',\n                          lambda: shutil.copyfile(archive, staged))\n                _progress(log, f\'{local.name}: unpacking on local SSD\', lambda: _unpack(staged, local))\n            stamp.write_text(signature)\n            (local.parent / (\'.\' + local.name + \'-\' + key + \'.saved\')).write_text(\n                _fingerprint(local, _files(local)))\n            log(f\'{local.name}: archive restore complete\')\n            return \'archive\'\n        except (OSError, ValueError, tarfile.TarError) as error:\n            log(f\'{local.name}: archive unavailable, falling back to the legacy cache ({error})\')\n    if legacy.is_dir():\n        try:\n            def migrate():\n                shutil.copytree(legacy, local, dirs_exist_ok=True, symlinks=True)\n            _progress(log, f\'{local.name}: first-time migration of the legacy Drive cache (slow when there are many small files)\', migrate)\n            if _files(local):\n                save_cache(local, archive_dir, key, log)\n                return \'legacy\'\n        except (OSError, ValueError, tarfile.TarError) as error:\n            log(f\'{local.name}: legacy cache migration failed, falling back to on-demand generation ({error})\')\n    log(f\'{local.name}: no persistent cache; the service will generate it on demand\')\n    return \'miss\'\n\n\ndef save_cache(local, archive_dir, key, log=print):\n    local, archive_dir = Path(local), Path(archive_dir)\n    files = _files(local)\n    if not files:\n        log(f\'{local.name}: no cache files, skipping archive\')\n        return False\n    fingerprint = _fingerprint(local, files)\n    saved = local.parent / (\'.\' + local.name + \'-\' + key + \'.saved\')\n    destination = archive_dir / key / (local.name + \'.tar\')\n    if saved.exists() and saved.read_text() == fingerprint and destination.is_file():\n        log(f\'{local.name}: content unchanged, skipping upload\')\n        return False\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    temporary = destination.with_name(destination.name + \'.\' + uuid.uuid4().hex + \'.tmp\')\n    try:\n        with tempfile.TemporaryDirectory(dir=local.parent, prefix=\'.cache-save-\') as tmp:\n            staged = Path(tmp) / \'cache.tar\'\n            def pack():\n                with tarfile.open(staged, \'w:\') as tar:\n                    for path in files:\n                        tar.add(path, arcname=str(path.relative_to(local)), recursive=False)\n            _progress(log, f\'{local.name}: packing {len(files)} files locally\', pack)\n            _progress(log, f\'{local.name}: saving archive to Drive ({staged.stat().st_size/1e6:.1f} MB)\',\n                      lambda: shutil.copyfile(staged, temporary))\n            os.replace(temporary, destination)\n        saved.write_text(fingerprint)\n        stat = destination.stat()\n        (local.parent / (\'.\' + local.name + \'-\' + key + \'.restored\')).write_text(\n            f\'{stat.st_size}:{stat.st_mtime_ns}\')\n        log(f\'{local.name}: Drive archive updated\')\n        return True\n    finally:\n        temporary.unlink(missing_ok=True)\n', encoding="utf-8")
if str(D3_WORK) not in sys.path:
    sys.path.insert(0, str(D3_WORK))
from colab_cache import restore_cache, save_cache

CACHE_KEY = f"sglomni-{SGL_OMNI_VER}-py312-{VARIANT}-sm{GPU_ARCH}"
CACHE_ARCHIVES = Path(DATA_DIR) / "cache-archives"
CACHE_PAIRS = (
    (FLASHINFER_CACHE_DIR, FLASHINFER_LEGACY_DIR),
    (TRITON_CACHE_DIR, TRITON_LEGACY_DIR),
    (INDUCTOR_CACHE_DIR, INDUCTOR_LEGACY_DIR),
)
for _local_c, _legacy_c in CACHE_PAIRS:
    try:
        restore_cache(_local_c, _legacy_c, CACHE_ARCHIVES, CACHE_KEY, dlog)
    except (OSError, ValueError) as err:
        dlog(f"Cache restore warning ({_local_c.name}): {err}; continuing with local cache")

dlog("Restoring registered voice samples from Drive")
subprocess.run(["cp", "-rn", f"{DRIVE_VOICES_DIR}/.", f"{SPEAKERS_DIR}/"], check=False)
dlog(
    f"GPU: {GPU_NAME} ({GPU_VRAM_MIB} MiB, sm_{GPU_ARCH}) | model={MODEL_REPO} ({VARIANT}) | "
    f"engine_mem={ENGINE_GPU_MEM} vocoder_mem={VOCODER_GPU_MEM} max_reqs={MAX_RUNNING_REQS} batch_max={BATCH_MAX_ITEMS}"
)

if not shutil.which("ffmpeg") or not shutil.which("sox"):
    subprocess.run(
        "apt-get update -qq && apt-get install -y -qq ffmpeg sox libsox-fmt-all >/dev/null 2>&1",
        shell=True,
        check=False,
    )
if not shutil.which("uv"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "uv"], check=True)

def run_logged(cmd, log_path, env=None, cwd=None, check=True):
    with open(log_path, "a", encoding="utf-8") as f:
        f.write(f"\n$ {' '.join(str(x) for x in cmd)}\n")
        f.flush()
        res = subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT, env=env, cwd=cwd, text=True)
    if check and res.returncode != 0:
        tail = "".join(open(log_path, errors="replace").readlines()[-45:])
        raise RuntimeError(f"Command failed ({res.returncode}): {' '.join(str(x) for x in cmd[:4])}\n{tail}")
    return res

def prepare_env():
    env_log = D3_WORK / "env-setup.log"
    marker = VENV_DIR / f".ct_drama3_env_{SGL_OMNI_VER}_ready"
    py_bin = VENV_DIR / "bin" / "python"
    sgl_bin = VENV_DIR / "bin" / "sgl-omni"
    if marker.is_file() and py_bin.is_file() and sgl_bin.is_file():
        dlog(f"Reusing existing sglang-omni=={SGL_OMNI_VER} virtualenv at {VENV_DIR}")
        return VENV_DIR

    dlog(f"Installing sglang-omni[fishaudio]=={SGL_OMNI_VER} in Python 3.12 venv (log: {env_log})...")
    uv_bin = shutil.which("uv") or "uv"
    overrides_file = D3_WORK / "uv-overrides.txt"
    # descript-audiotools pins protobuf<3.20 while sglang/nvidia-cutlass-dsl requires protobuf>=4.21.12;
    # sglang-omni upstream pyproject.toml resolves this via override-dependencies protobuf>=6.31.1,<7.0.0.
    overrides_file.write_text("protobuf>=6.31.1,<7.0.0\n", encoding="utf-8")
    run_logged([uv_bin, "venv", "--python", "3.12", str(VENV_DIR), "--clear"], env_log)
    run_logged(
        [
            uv_bin, "pip", "install",
            "--python", str(py_bin),
            "--prerelease=allow",
            "--overrides", str(overrides_file),
            f"sglang-omni[fishaudio]=={SGL_OMNI_VER}",
            "huggingface_hub",
            "hf_xet",
        ],
        env_log,
    )
    run_logged(
        [str(py_bin), "-c", "import torch, sglang, sglang_omni, dac, audiotools; print('imports-ok', torch.__version__)"],
        env_log,
    )
    marker.write_text(f"{SGL_OMNI_VER}\n", encoding="utf-8")
    dlog(f"sglang-omni[fishaudio]=={SGL_OMNI_VER} virtualenv ready")
    return VENV_DIR

def _cleanup_hf_temp():
    for p in ("/root/.cache/huggingface/xet", "/root/.cache/huggingface/hub"):
        shutil.rmtree(p, ignore_errors=True)
    for mp in (MODEL_DIR / ".cache", CODEC_DIR / ".cache"):
        shutil.rmtree(mp, ignore_errors=True)

def _ensure_default_ref_wav():
    if DEFAULT_REF_WAV.is_file() and DEFAULT_REF_WAV.stat().st_size > 1024:
        return
    drive_copy = DRIVE_VOICES_DIR / "default_ref.wav"
    if drive_copy.is_file() and drive_copy.stat().st_size > 1024:
        shutil.copyfile(drive_copy, DEFAULT_REF_WAV)
        return
    for ref_url in (
        "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen3-TTS-Repo/clone.wav",
        "https://raw.githubusercontent.com/vllm-project/vllm-omni/main/tests/assets/indextts2/ref_audio.wav",
    ):
        try:
            with urllib.request.urlopen(ref_url, timeout=60) as _src, open(DEFAULT_REF_WAV, "wb") as _dst:
                shutil.copyfileobj(_src, _dst)
            if DEFAULT_REF_WAV.is_file() and DEFAULT_REF_WAV.stat().st_size > 1024:
                break
        except Exception:
            pass
    if not DEFAULT_REF_WAV.is_file() or DEFAULT_REF_WAV.stat().st_size <= 1024:
        import math, struct, wave
        sr = 44100
        n = sr * 3
        with wave.open(str(DEFAULT_REF_WAV), "wb") as wf:
            wf.setnchannels(1)
            wf.setsampwidth(2)
            wf.setframerate(sr)
            samples = [int(10000 * math.sin(2 * math.pi * 220 * i / sr)) for i in range(n)]
            wf.writeframes(struct.pack(f"<{n}h", *samples))
    if not drive_copy.is_file():
        shutil.copyfile(DEFAULT_REF_WAV, drive_copy)

def prepare_model():
    dl_log = D3_WORK / "model-download.log"
    _ensure_default_ref_wav()
    marker = MODEL_DIR / ".ct_model_ready"
    codec_file = CODEC_DIR / "codec.pth"
    has_weights = any(MODEL_DIR.glob("*.safetensors"))
    if marker.is_file() and (MODEL_DIR / "config.json").is_file() and has_weights and codec_file.is_file():
        dlog(f"Reusing local SSD model {MODEL_DIR} and codec {codec_file}")
        return MODEL_DIR

    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub", "hf_xet"], check=True)
    _cleanup_hf_temp()
    free_gb = shutil.disk_usage("/content").free / 1e9
    dlog(f"Downloading {MODEL_REPO} + {CODEC_REPO} codec -> local SSD {D3_WORK} (free SSD: {free_gb:.1f} GB; log: {dl_log})...")
    dl_env = {**os.environ, "HF_XET_CHUNK_CACHE_SIZE_BYTES": "0", "HF_HUB_ENABLE_HF_TRANSFER": "0"}
    dl_env.pop("HF_XET_HIGH_PERFORMANCE", None)
    dl_script = r"""
import shutil, sys
from pathlib import Path
from huggingface_hub import hf_hub_download, snapshot_download

repo_id, model_dir, codec_repo, codec_rev, codec_dir = sys.argv[1:6]
model_path = Path(model_dir)
codec_path = Path(codec_dir)
model_path.mkdir(parents=True, exist_ok=True)
codec_path.mkdir(parents=True, exist_ok=True)

snapshot_download(
    repo_id=repo_id,
    local_dir=str(model_path),
    max_workers=8,
)
if not (codec_path / "codec.pth").is_file():
    hf_hub_download(
        repo_id=codec_repo,
        filename="codec.pth",
        revision=codec_rev,
        local_dir=str(codec_path),
    )
for p in ("/root/.cache/huggingface/xet", "/root/.cache/huggingface/hub"):
    shutil.rmtree(p, ignore_errors=True)
for mp in (model_path / ".cache", codec_path / ".cache"):
    shutil.rmtree(mp, ignore_errors=True)
"""
    try:
        run_logged(
            [sys.executable, "-c", dl_script, MODEL_REPO, str(MODEL_DIR), CODEC_REPO, CODEC_REVISION, str(CODEC_DIR)],
            dl_log,
            env=dl_env,
        )
    finally:
        _cleanup_hf_temp()

    if not (MODEL_DIR / "config.json").is_file() or not any(MODEL_DIR.glob("*.safetensors")) or not codec_file.is_file():
        raise RuntimeError(f"Model/codec download incomplete in {MODEL_DIR} / {CODEC_DIR}")
    marker.write_text(MODEL_REPO + "\n", encoding="utf-8")
    total_gb = (
        sum(p.stat().st_size for p in MODEL_DIR.rglob("*") if p.is_file())
        + codec_file.stat().st_size
    ) / 1e9
    dlog(f"Drama 3 weights + S2-Pro codec ready on local SSD ({total_gb:.2f} GB)")
    return MODEL_DIR

with ThreadPoolExecutor(max_workers=2) as pool:
    f_env = pool.submit(prepare_env)
    f_mod = pool.submit(prepare_model)
    while not (f_env.done() and f_mod.done()):
        for fut in (f_env, f_mod):
            if fut.done() and fut.exception():
                raise fut.exception()
        free_gb = shutil.disk_usage("/content").free / 1e9
        dlog(
            f"Setup heartbeat | env={'done' if f_env.done() else 'running'} "
            f"model={'done' if f_mod.done() else 'running'} | free_ssd={free_gb:.1f} GB"
        )
        time.sleep(25)
    VENV_DIR = f_env.result()
    MODEL_DIR = f_mod.result()

dlog(f"Setup complete: venv={VENV_DIR} model={MODEL_DIR} codec={CODEC_DIR} drive_data={DATA_DIR}")


In [ ]:
# @title 2. Service -- SGLang-Omni Fish Audio Drama 3 on 127.0.0.1:8000 (Bearer Auth + Drive Voices & Batch TTS)
(D3_WORK / "colab_asgi_common.py").write_text('"""Shared security, Drive path containment, and audio helpers for Colab ASGI middlewares."""\nfrom __future__ import annotations\n\nimport base64\nimport glob as glob_mod\nimport os\nimport re\nimport secrets\nimport subprocess\nimport time\nfrom pathlib import Path\nfrom typing import Any\n\nAUDIO_EXTS = (\n    ".wav", ".flac", ".mp3", ".ogg", ".m4a", ".aac", ".opus", ".webm", ".mp4", ".mkv", ".wma", ".mpga", ".mpeg",\n)\nDEFAULT_SKIP_DIRS = {\n    ".git", "node_modules", "__pycache__", ".venv", "venv", "aux-venv", "custom_nodes",\n    "models", ".runtimes", "Takeout", ".Trash", ".ipynb_checkpoints",\n    "flashinfer-cache", "triton-cache", "vllm-cache", "cache-archives", "kernel-archives",\n}\nVOICE_NAME_RE = re.compile(r"^[a-z0-9][a-z0-9._-]{0,63}$")\n\n\ndef validate_voice_name(name: str) -> str:\n    """Validate and normalize a custom voice identifier against path traversal."""\n    vname = (name or "").strip().lower()\n    if not vname or ".." in vname or not VOICE_NAME_RE.fullmatch(vname):\n        raise ValueError(f"Invalid voice name \'{name}\': use 1-64 lowercase [a-z0-9._-] characters without \'..\'")\n    return vname\n\n\ndef _get_header(headers: Any, key: str) -> str:\n    val = headers.get(key)\n    if val:\n        return str(val)\n    if isinstance(headers, dict):\n        low = key.lower()\n        for k, v in headers.items():\n            if str(k).lower() == low and v:\n                return str(v)\n    return ""\n\n\ndef verify_bearer_auth(headers: Any, api_token: str) -> bool:\n    """Fail-closed constant-time Bearer / X-API-Key authentication check."""\n    if not api_token:\n        return False\n    expected_bearer = f"Bearer {api_token}"\n    auth = _get_header(headers, "authorization")\n    if auth and secrets.compare_digest(auth.encode("utf-8"), expected_bearer.encode("utf-8")):\n        return True\n    xkey = _get_header(headers, "x-api-key")\n    if xkey and secrets.compare_digest(xkey.encode("utf-8"), api_token.encode("utf-8")):\n        return True\n    return False\n\n\ndef sanitize_header_param(value: str) -> str:\n    """Strip CR/LF/NUL and escape quotes/backslashes for Content-Disposition parameters."""\n    cleaned = (value or "").replace("\\r", "").replace("\\n", "").replace("\\x00", "")\n    return cleaned.replace("\\\\", "\\\\\\\\").replace(\'"\', \'\\\\"\')\n\n\ndef build_multipart_form(\n    fields: list[tuple[str, str]],\n    files: list[tuple[str, str, str, bytes]] | None = None,\n) -> tuple[str, bytes]:\n    """Build a safe multipart/form-data body returning (content_type, body_bytes)."""\n    boundary = f"----ctform{secrets.token_hex(16)}"\n    parts: list[bytes] = []\n    for key, val in fields:\n        safe_key = sanitize_header_param(str(key))\n        parts.append(\n            f\'--{boundary}\\r\\nContent-Disposition: form-data; name="{safe_key}"\\r\\n\\r\\n{val}\\r\\n\'.encode("utf-8")\n        )\n    for field_name, filename, content_type, data in files or []:\n        safe_field = sanitize_header_param(str(field_name))\n        safe_filename = sanitize_header_param(str(filename))\n        safe_ct = (content_type or "application/octet-stream").replace("\\r", "").replace("\\n", "").replace("\\x00", "")\n        parts.append(\n            (\n                f"--{boundary}\\r\\n"\n                f\'Content-Disposition: form-data; name="{safe_field}"; filename="{safe_filename}"\\r\\n\'\n                f"Content-Type: {safe_ct}\\r\\n\\r\\n"\n            ).encode("utf-8")\n            + data\n            + b"\\r\\n"\n        )\n    parts.append(f"--{boundary}--\\r\\n".encode("utf-8"))\n    return f"multipart/form-data; boundary={boundary}", b"".join(parts)\n\n\ndef allowed_drive_roots(\n    data_dir: Path,\n    mydrive_root: Path = Path("/content/drive/MyDrive"),\n) -> tuple[Path, ...]:\n    """Return resolved root directories that Drive/data operations may access."""\n    roots: list[Path] = []\n    if mydrive_root.is_dir():\n        roots.append(mydrive_root.resolve())\n    roots.append(data_dir.resolve())\n    return tuple(dict.fromkeys(roots))\n\n\ndef _ensure_within_roots(candidate: Path, roots: tuple[Path, ...], raw_spec: str) -> Path:\n    resolved = candidate.resolve()\n    if not any(resolved.is_relative_to(root) for root in roots):\n        raise ValueError(f"Path \'{raw_spec}\' escapes allowed Drive/data directory")\n    return resolved\n\n\ndef resolve_drive_path(\n    raw_spec: str,\n    *,\n    data_dir: Path,\n    default_subdir: str | None = None,\n    mydrive_root: Path = Path("/content/drive/MyDrive"),\n    drive_root: Path = Path("/content/drive"),\n) -> Path:\n    """Resolve a Drive or service-data path strictly inside allowed roots."""\n    if "\\x00" in (raw_spec or ""):\n        raise ValueError("Invalid NUL byte in path")\n    roots = allowed_drive_roots(data_dir, mydrive_root)\n    spec = (raw_spec or "").strip()\n\n    is_file_scheme = spec.startswith("file://")\n    for prefix in ("drive://", "gdrive://", "file://"):\n        if spec.startswith(prefix):\n            spec = spec[len(prefix):]\n            break\n\n    if is_file_scheme:\n        candidate = Path("/" + spec.lstrip("/"))\n        return _ensure_within_roots(candidate, roots, raw_spec)\n\n    had_leading_slash = spec.startswith("/")\n    clean = spec.lstrip("/")\n    if not clean:\n        if default_subdir and (data_dir / default_subdir).is_dir():\n            return _ensure_within_roots(data_dir / default_subdir, roots, raw_spec)\n        return roots[0]\n\n    if had_leading_slash and not clean.startswith(("content/drive/", "drive/", "MyDrive/", "MyDrive")):\n        # Allow explicit absolute path ONLY if it already lies inside an allowed root (e.g. DATA_DIR in --no-drive mode)\n        return _ensure_within_roots(Path("/" + clean), roots, raw_spec)\n\n    candidates: list[Path] = []\n    if clean.startswith("content/drive/"):\n        candidates.append(Path("/") / clean)\n    elif clean.startswith("drive/"):\n        candidates.append(drive_root.parent / clean)\n    elif clean.startswith("MyDrive/"):\n        candidates.append(drive_root / clean)\n    elif clean == "MyDrive":\n        candidates.append(mydrive_root)\n    elif clean.startswith("colab-llm/") and mydrive_root.is_dir():\n        candidates.append(mydrive_root / clean)\n        candidates.append(data_dir / clean)\n    else:\n        first_seg = clean.split("/", 1)[0]\n        already_prefixed = bool(\n            default_subdir\n            and (\n                clean == default_subdir\n                or clean.startswith(f"{default_subdir}/")\n                or (first_seg != clean and (data_dir / first_seg).is_dir())\n            )\n        )\n        if default_subdir and not already_prefixed:\n            candidates.append(data_dir / default_subdir / clean)\n        candidates.append(data_dir / clean)\n        if default_subdir and already_prefixed:\n            candidates.append(data_dir / default_subdir / clean)\n        if mydrive_root.is_dir():\n            candidates.append(mydrive_root / clean)\n\n    checked: list[Path] = []\n    for cand in candidates:\n        resolved = _ensure_within_roots(cand, roots, raw_spec)\n        checked.append(resolved)\n        if resolved.exists():\n            return resolved\n    if not checked:\n        raise ValueError(f"Path \'{raw_spec}\' could not be resolved inside allowed Drive/data directory")\n    return checked[0]\n\n\ndef rel_to_mydrive(p: Path, mydrive_root: Path = Path("/content/drive/MyDrive")) -> str:\n    try:\n        return str(p.resolve().relative_to(mydrive_root.resolve()))\n    except Exception:\n        return str(p)\n\n\ndef list_drive_audio_files(\n    spec: str,\n    *,\n    data_dir: Path,\n    default_subdir: str | None = None,\n    recursive: bool = True,\n    mydrive_root: Path = Path("/content/drive/MyDrive"),\n    drive_root: Path = Path("/content/drive"),\n    skip_dirs: set[str] | None = None,\n) -> list[dict[str, Any]]:\n    """List audio files matching a Drive path or glob strictly inside allowed roots."""\n    if "\\x00" in (spec or ""):\n        raise ValueError("Invalid NUL byte in path")\n    roots = allowed_drive_roots(data_dir, mydrive_root)\n    ignored = skip_dirs if skip_dirs is not None else DEFAULT_SKIP_DIRS\n    raw = (spec or "").strip()\n    is_file_scheme = raw.startswith("file://")\n    for prefix in ("drive://", "gdrive://", "file://"):\n        if raw.startswith(prefix):\n            raw = raw[len(prefix):]\n            break\n\n    has_glob = any(ch in raw for ch in ("*", "?", "["))\n    matched_paths: list[Path] = []\n\n    if has_glob:\n        if is_file_scheme:\n            bases = [Path("/" + raw.lstrip("/"))]\n        else:\n            clean = raw.lstrip("/")\n            if raw.startswith("/") and not clean.startswith(("content/drive/", "drive/", "MyDrive/")):\n                bases = [Path("/" + clean)]\n            elif clean.startswith("content/drive/"):\n                bases = [Path("/") / clean]\n            elif clean.startswith("MyDrive/"):\n                bases = [drive_root / clean]\n            else:\n                bases = []\n                if default_subdir:\n                    bases.append(data_dir / default_subdir / clean)\n                bases.append(data_dir / clean)\n                if mydrive_root.is_dir():\n                    bases.append(mydrive_root / clean)\n\n        # Ensure static prefix before wildcard does not escape allowed roots\n        for pattern_path in bases:\n            prefix_parts: list[str] = []\n            for part in pattern_path.parts:\n                if any(ch in part for ch in ("*", "?", "[")):\n                    break\n                prefix_parts.append(part)\n            if prefix_parts:\n                _ensure_within_roots(Path(*prefix_parts), roots, spec)\n\n        seen: set[str] = set()\n        for pattern_path in bases:\n            for hit in sorted(glob_mod.glob(str(pattern_path), recursive=True)):\n                hp = Path(hit).resolve()\n                if not any(hp.is_relative_to(r) for r in roots):\n                    continue\n                if hp.is_file() and hp.suffix.lower() in AUDIO_EXTS and str(hp) not in seen:\n                    seen.add(str(hp))\n                    matched_paths.append(hp)\n    else:\n        target = resolve_drive_path(\n            spec,\n            data_dir=data_dir,\n            default_subdir=default_subdir,\n            mydrive_root=mydrive_root,\n            drive_root=drive_root,\n        )\n        if target.is_file():\n            if target.suffix.lower() in AUDIO_EXTS:\n                matched_paths.append(target)\n        elif target.is_dir():\n            if not recursive:\n                for hp in sorted(target.iterdir()):\n                    resolved = hp.resolve()\n                    if any(resolved.is_relative_to(r) for r in roots) and resolved.is_file() and resolved.suffix.lower() in AUDIO_EXTS:\n                        matched_paths.append(resolved)\n            else:\n                for root_dir, dirnames, filenames in os.walk(target):\n                    dirnames[:] = sorted(d for d in dirnames if d not in ignored and not d.startswith("."))\n                    for fn in sorted(filenames):\n                        hp = (Path(root_dir) / fn).resolve()\n                        if any(hp.is_relative_to(r) for r in roots) and hp.is_file() and hp.suffix.lower() in AUDIO_EXTS:\n                            matched_paths.append(hp)\n\n    items: list[dict[str, Any]] = []\n    for p in matched_paths:\n        st = p.stat()\n        items.append({\n            "drive_path": str(p),\n            "rel_path": rel_to_mydrive(p, mydrive_root),\n            "name": p.name,\n            "size_bytes": st.st_size,\n            "size_mb": round(st.st_size / 1e6, 2),\n            "mtime": int(st.st_mtime),\n        })\n    return items\n\n\ndef audio_path_to_wav_data_url(\n    audio_path: Path,\n    max_duration_sec: float = 24.0,\n    sample_rate: int = 24000,\n    timeout: int = 60,\n) -> str:\n    """Decode a Drive audio file to a mono PCM16 WAV data URL with bounded ffmpeg timeout."""\n    if not audio_path.is_file():\n        raise FileNotFoundError(f"Audio file not found: {audio_path}")\n    cmd = [\n        "ffmpeg", "-nostdin", "-v", "error",\n        "-t", f"{max_duration_sec:.1f}",\n        "-i", str(audio_path),\n        "-vn", "-ac", "1", "-ar", str(sample_rate), "-c:a", "pcm_s16le", "-f", "wav", "pipe:1",\n    ]\n    try:\n        res = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, timeout=timeout, check=False)\n    except OSError as os_err:\n        if audio_path.suffix.lower() == ".wav":\n            b64 = base64.b64encode(audio_path.read_bytes()).decode("ascii")\n            return f"data:audio/wav;base64,{b64}"\n        raise RuntimeError(f"ffmpeg failed decoding {audio_path.name}: {os_err}") from os_err\n    if res.returncode != 0 or not res.stdout:\n        if audio_path.suffix.lower() == ".wav":\n            b64 = base64.b64encode(audio_path.read_bytes()).decode("ascii")\n            return f"data:audio/wav;base64,{b64}"\n        err = res.stderr.decode("utf-8", errors="replace").strip()\n        raise RuntimeError(f"ffmpeg failed decoding {audio_path.name}: {err}")\n    b64 = base64.b64encode(res.stdout).decode("ascii")\n    return f"data:audio/wav;base64,{b64}"\n\n\ndef save_output_audio_to_drive(\n    audio_bytes: bytes,\n    spec: str,\n    *,\n    data_dir: Path,\n    output_dir: Path,\n    prefix: str,\n    fmt: str = "wav",\n    mydrive_root: Path = Path("/content/drive/MyDrive"),\n    drive_root: Path = Path("/content/drive"),\n) -> str:\n    """Persist synthesized audio bytes safely inside Drive or service output directory."""\n    output_dir.mkdir(parents=True, exist_ok=True)\n    ext = "." + re.sub(r"[^a-z0-9]", "", (fmt or "wav").lower())\n    if not ext or ext == ".":\n        ext = ".wav"\n    if spec and spec not in ("auto", "true", "True", "1"):\n        dest = resolve_drive_path(\n            spec,\n            data_dir=data_dir,\n            mydrive_root=mydrive_root,\n            drive_root=drive_root,\n        )\n        if dest.is_dir() or spec.endswith("/"):\n            dest.mkdir(parents=True, exist_ok=True)\n            dest = dest / f"{prefix}-{time.strftime(\'%Y%m%d-%H%M%S\')}{ext}"\n        else:\n            dest.parent.mkdir(parents=True, exist_ok=True)\n            if not dest.suffix:\n                dest = dest.with_suffix(ext)\n    else:\n        dest = output_dir / f"{prefix}-{time.strftime(\'%Y%m%d-%H%M%S\')}{ext}"\n    dest.write_bytes(audio_bytes)\n    return str(dest)\n\n\ndef tail_service_log(\n    log_file: Path = Path("/content/ct/service.log"),\n    lines: int = 120,\n) -> dict[str, Any]:\n    n_lines = max(1, min(int(lines), 2000))\n    entries = log_file.read_text(encoding="utf-8", errors="replace").splitlines()[-n_lines:] if log_file.is_file() else []\n    return {"lines": len(entries), "log": "\\n".join(entries)}\n', encoding="utf-8")
(D3_WORK / "ct_drama3_middleware.py").write_text('"""Authenticated ASGI middleware, Google Drive voice/output I/O, and launcher wrapper for Fish Audio Drama 3."""\nfrom __future__ import annotations\n\nimport asyncio\nimport base64\nfrom http import HTTPStatus\nimport json\nimport os\nfrom pathlib import Path\nimport shutil\nimport subprocess\nimport sys\nimport tempfile\nfrom typing import Any\nimport urllib.error\nfrom urllib.parse import parse_qs, unquote\nimport urllib.request\n\nfrom colab_asgi_common import (\n    audio_path_to_wav_data_url,\n    build_multipart_form,\n    list_drive_audio_files,\n    resolve_drive_path,\n    sanitize_header_param,\n    save_output_audio_to_drive,\n    tail_service_log,\n    validate_voice_name,\n    verify_bearer_auth,\n)\n\ntry:\n    from starlette.datastructures import Headers\n    from starlette.responses import JSONResponse, Response\n    from starlette.types import ASGIApp, Message, Receive, Scope, Send\nexcept ImportError:  # pragma: no cover - local unit-test fallback without starlette\n    ASGIApp = Message = Receive = Scope = Send = Any  # type: ignore[assignment,misc]\n\n    class Headers:  # type: ignore[no-redef]\n        def __init__(self, scope: dict[str, Any]) -> None:\n            self._d = {k.decode("latin1").lower(): v.decode("latin1") for k, v in scope.get("headers", [])}\n\n        def get(self, key: str, default: str = "") -> str:\n            return self._d.get(key.lower(), default)\n\n    class Response:  # type: ignore[no-redef]\n        def __init__(self, content: bytes = b"", status_code: int = 200, media_type: str | None = None) -> None:\n            self.body = content\n            self.status_code = int(status_code)\n            self.media_type = media_type\n\n        async def __call__(self, scope: Scope, receive: Receive, send: Send) -> None:\n            hdrs = [(b"content-length", str(len(self.body)).encode("ascii"))]\n            if self.media_type:\n                hdrs.append((b"content-type", self.media_type.encode("ascii")))\n            await send({"type": "http.response.start", "status": self.status_code, "headers": hdrs})\n            await send({"type": "http.response.body", "body": self.body, "more_body": False})\n\n    class JSONResponse(Response):  # type: ignore[no-redef]\n        def __init__(self, content: Any, status_code: int = 200) -> None:\n            super().__init__(json.dumps(content).encode("utf-8"), status_code=status_code, media_type="application/json")\n\nAPI_TOKEN = os.environ.get("CT_API_TOKEN", "")\nSERVICE_PORT = int((os.environ.get("CT_SERVICE_PORT") or "").strip() or "8000")\nSERVED_MODEL = (os.environ.get("CT_DRAMA3_MODEL") or "fishaudio/drama-3").strip()\nMODEL_ALIAS = (os.environ.get("CT_DRAMA3_ALIAS") or "drama3").strip()\nMAX_REF_SEC = float((os.environ.get("CT_DRAMA3_MAX_REF_SEC") or "").strip() or "60.0")\nSAMPLE_RATE = 44100\n\nDATA_DIR = Path(os.environ.get("CT_DATA_DIR", "/content/drive/MyDrive/colab-llm/drama3"))\nSPEAKERS_DIR = Path(os.environ.get("SPEAKER_SAMPLES_DIR", "/content/drama3/speakers"))\nDEFAULT_REF_WAV = Path(os.environ.get("CT_DEFAULT_REF_WAV", "/content/drama3/default_ref.wav"))\nDEFAULT_REF_TEXT = (os.environ.get("CT_DEFAULT_REF_TEXT") or "").strip()\nDRIVE_ROOT = Path("/content/drive")\nMYDRIVE_ROOT = Path("/content/drive/MyDrive")\nDRIVE_VOICES_DIR = DATA_DIR / "voices"\nDRIVE_OUTPUT_DIR = DATA_DIR / "outputs"\n\nMODEL_ALIASES = frozenset({\n    "",\n    "default",\n    "drama3",\n    "drama-3",\n    "drama-3-bf16",\n    "drama-3-int4",\n    "fishaudio/drama-3",\n    "fishaudio/drama-3-int4",\n    "tts-1",\n    "tts-1-hd",\n    "gpt-4o-mini-tts",\n})\n\nUNSUPPORTED_SPEECH_KEYS = (\n    "language",\n    "instructions",\n    "seed",\n    "extra_params",\n    "x_vector_only_mode",\n    "stream_codec_output",\n    "suppress_bootstrap_silence",\n    "token_count",\n    "duration_tokens",\n    "initial_codec_chunk_frames",\n)\n\nBLOCKED_ADMIN_PREFIXES = (\n    "/pause_generation",\n    "/continue_generation",\n    "/update_weights",\n    "/update_weights_from_disk",\n    "/update_weights_from_tensor",\n    "/update_weights_from_distributed",\n    "/init_weights_update_group",\n    "/destroy_weights_update_group",\n    "/weights_checker",\n)\n\nCORS_HEADERS: tuple[tuple[bytes, bytes], ...] = (\n    (b"access-control-allow-origin", b"*"),\n    (b"access-control-allow-methods", b"GET, POST, DELETE, OPTIONS"),\n    (\n        b"access-control-allow-headers",\n        b"Authorization, Content-Type, X-API-Key, X-Save-Drive",\n    ),\n    (b"access-control-expose-headers", b"X-Saved-Drive-Path, X-Audio-Sample-Rate"),\n    (b"access-control-max-age", b"86400"),\n)\n\n\ndef _verify_auth(headers: Any) -> bool:\n    return verify_bearer_auth(headers, API_TOKEN)\n\n\ndef _resolve_path(raw_spec: str, default_subdir: str | None = "voices") -> Path:\n    return resolve_drive_path(\n        raw_spec,\n        data_dir=DATA_DIR,\n        default_subdir=default_subdir,\n        mydrive_root=MYDRIVE_ROOT,\n        drive_root=DRIVE_ROOT,\n    )\n\n\ndef _validate_save_drive_spec(spec: str) -> str:\n    cleaned = (spec or "").strip()\n    if not cleaned:\n        return ""\n    if cleaned not in ("auto", "true", "True", "1"):\n        resolve_drive_path(\n            cleaned,\n            data_dir=DATA_DIR,\n            mydrive_root=MYDRIVE_ROOT,\n            drive_root=DRIVE_ROOT,\n        )\n    return cleaned\n\n\ndef _sync_speakers_to_drive(deleted_voice: str | None = None) -> None:\n    try:\n        DRIVE_VOICES_DIR.mkdir(parents=True, exist_ok=True)\n        if deleted_voice:\n            try:\n                vname = validate_voice_name(deleted_voice)\n                if not (SPEAKERS_DIR / f"{vname}.safetensors").is_file():\n                    (DRIVE_VOICES_DIR / f"{vname}.safetensors").unlink(missing_ok=True)\n            except ValueError:\n                pass\n        if SPEAKERS_DIR.is_dir():\n            for sf in SPEAKERS_DIR.glob("*.safetensors"):\n                dst = DRIVE_VOICES_DIR / sf.name\n                if not dst.is_file() or sf.stat().st_mtime > dst.stat().st_mtime:\n                    shutil.copy2(sf, dst)\n    except Exception:\n        pass\n\n\ndef _resolve_ref_audio_value(raw_val: str) -> str:\n    ra = raw_val.strip()\n    if ra.startswith(("http://", "https://", "data:")):\n        return ra\n    ap = _resolve_path(ra)\n    return audio_path_to_wav_data_url(ap, max_duration_sec=MAX_REF_SEC, sample_rate=SAMPLE_RATE)\n\n\ndef _normalize_speech_item(item: dict[str, Any], *, is_batch_item: bool = False) -> bool:\n    changed = False\n\n    model_val = item.get("model")\n    if isinstance(model_val, str):\n        m_clean = model_val.strip()\n        if m_clean.lower() in MODEL_ALIASES or m_clean == MODEL_ALIAS:\n            if model_val != SERVED_MODEL:\n                item["model"] = SERVED_MODEL\n                changed = True\n    elif model_val is None and not is_batch_item:\n        item["model"] = SERVED_MODEL\n        changed = True\n\n    for unsupported_key in UNSUPPORTED_SPEECH_KEYS:\n        if unsupported_key in item:\n            item.pop(unsupported_key, None)\n            changed = True\n\n    ref_audio = item.get("ref_audio")\n    if isinstance(ref_audio, str) and ref_audio.strip():\n        rewritten = _resolve_ref_audio_value(ref_audio)\n        if rewritten != ref_audio:\n            item["ref_audio"] = rewritten\n            changed = True\n\n    refs = item.get("references")\n    if isinstance(refs, list):\n        for ref_entry in refs:\n            if not isinstance(ref_entry, dict):\n                continue\n            for key_name in ("audio_path", "ref_audio", "audio"):\n                raw_ref = ref_entry.get(key_name)\n                if isinstance(raw_ref, str) and raw_ref.strip():\n                    rewritten = _resolve_ref_audio_value(raw_ref)\n                    if key_name != "audio_path":\n                        ref_entry.pop(key_name, None)\n                    ref_entry["audio_path"] = rewritten\n                    changed = True\n                    break\n\n    return changed\n\n\ndef _prepare_speech_request(raw_body: bytes, *, is_batch: bool = False) -> tuple[bytes, str | None, str]:\n    if not raw_body:\n        return raw_body, None, "wav"\n    try:\n        payload = json.loads(raw_body)\n    except Exception:\n        return raw_body, None, "wav"\n    if not isinstance(payload, dict):\n        return raw_body, None, "wav"\n\n    changed = False\n    save_drive_spec: str | None = None\n    resp_fmt = str(payload.get("response_format") or "wav").strip().lower() or "wav"\n\n    for save_key in ("save_to_drive", "save_drive_path"):\n        if save_key in payload:\n            val = payload.pop(save_key)\n            changed = True\n            if val:\n                raw_spec = str(val) if not isinstance(val, bool) else "auto"\n                save_drive_spec = _validate_save_drive_spec(raw_spec)\n\n    if _normalize_speech_item(payload, is_batch_item=False):\n        changed = True\n\n    if is_batch and isinstance(payload.get("items"), list):\n        for entry in payload["items"]:\n            if isinstance(entry, dict) and _normalize_speech_item(entry, is_batch_item=True):\n                changed = True\n\n    if not changed:\n        return raw_body, save_drive_spec, resp_fmt\n    return json.dumps(payload, ensure_ascii=False).encode("utf-8"), save_drive_spec, resp_fmt\n\n\ndef _decode_base64_audio(raw_b64: str) -> bytes:\n    cleaned = raw_b64.strip()\n    if cleaned.startswith("data:") and "," in cleaned:\n        cleaned = cleaned.split(",", 1)[1]\n    return base64.b64decode(cleaned)\n\n\ndef _upload_drive_voice_sync(body: dict[str, Any]) -> tuple[int, dict[str, Any]]:\n    raw_name = str(body.get("name") or "").strip()\n    use_default_ref = bool(body.get("use_default_ref", False))\n    raw_path = str(body.get("drive_path") or body.get("audio") or body.get("path") or "").strip()\n    raw_b64 = str(body.get("audio_base64") or "").strip()\n    consent = str(body.get("consent") or "owner-consent").strip()\n    ref_text = body.get("ref_text")\n    speaker_desc = body.get("speaker_description")\n\n    if not raw_name:\n        return 400, {"error": "Field \'name\' is required"}\n    try:\n        name = validate_voice_name(raw_name)\n    except ValueError as err:\n        return 400, {"error": str(err)}\n\n    if not raw_path and not raw_b64 and not use_default_ref:\n        return 400, {"error": "Provide \'drive_path\', \'audio_base64\', or \'use_default_ref\'"}\n\n    temp_input: Path | None = None\n    try:\n        if use_default_ref:\n            ap = DEFAULT_REF_WAV\n            if not ap.is_file():\n                return 404, {"error": f"Default reference audio not found: {ap}"}\n            if not ref_text and DEFAULT_REF_TEXT:\n                ref_text = DEFAULT_REF_TEXT\n        elif raw_b64:\n            try:\n                raw_bytes = _decode_base64_audio(raw_b64)\n            except Exception as err:\n                return 400, {"error": f"Invalid audio_base64: {err}"}\n            with tempfile.NamedTemporaryFile(prefix="drama3-voice-", suffix=".bin", delete=False) as tf:\n                tf.write(raw_bytes)\n                temp_input = Path(tf.name)\n            ap = temp_input\n        else:\n            try:\n                ap = _resolve_path(raw_path)\n            except ValueError as err:\n                return 400, {"error": str(err)}\n            if not ap.is_file():\n                return 404, {"error": f"Audio file not found on Google Drive: {raw_path} (resolved to {ap})"}\n\n        cmd = [\n            "ffmpeg", "-nostdin", "-v", "error",\n            "-t", f"{MAX_REF_SEC:.1f}",\n            "-i", str(ap),\n            "-vn", "-ac", "1", "-ar", str(SAMPLE_RATE), "-c:a", "pcm_s16le", "-f", "wav", "pipe:1",\n        ]\n        try:\n            res = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, timeout=60, check=False)\n            wav_bytes = res.stdout if (res.returncode == 0 and res.stdout) else ap.read_bytes()\n        except OSError:\n            wav_bytes = ap.read_bytes()\n    finally:\n        if temp_input is not None:\n            temp_input.unlink(missing_ok=True)\n\n    fields: list[tuple[str, str]] = [("name", name), ("consent", consent)]\n    if ref_text:\n        fields.append(("ref_text", str(ref_text)))\n    if speaker_desc:\n        fields.append(("speaker_description", str(speaker_desc)))\n    content_type, body_bytes = build_multipart_form(\n        fields,\n        [("audio_sample", f"{name}.wav", "audio/wav", wav_bytes)],\n    )\n    req = urllib.request.Request(\n        f"http://127.0.0.1:{SERVICE_PORT}/v1/audio/voices",\n        data=body_bytes,\n        headers={\n            "Content-Type": content_type,\n            "Authorization": f"Bearer {API_TOKEN}",\n        },\n        method="POST",\n    )\n    try:\n        with urllib.request.urlopen(req, timeout=90) as resp:\n            out = json.loads(resp.read().decode("utf-8", errors="replace"))\n            _sync_speakers_to_drive()\n            return resp.status, out\n    except urllib.error.HTTPError as err:\n        raw_err = err.read().decode("utf-8", errors="replace")\n        try:\n            return err.code, json.loads(raw_err)\n        except Exception:\n            return err.code, {"error": raw_err}\n\n\ndef _with_cors_send(send: Send) -> Send:\n    existing_keys = {k.lower() for k, _ in CORS_HEADERS}\n\n    async def wrapped_send(message: Message) -> None:\n        if message["type"] == "http.response.start":\n            hdrs = [\n                (k, v) for (k, v) in message.get("headers", []) if k.lower() not in existing_keys\n            ]\n            hdrs.extend(CORS_HEADERS)\n            message = {**message, "headers": hdrs}\n        await send(message)\n\n    return wrapped_send\n\n\nclass Drama3Middleware:\n    """Fail-closed Bearer authentication, CORS, and Google Drive I/O middleware for SGLang-Omni Drama 3."""\n\n    def __init__(self, app: ASGIApp) -> None:\n        self.app = app\n\n    async def __call__(self, scope: Scope, receive: Receive, send: Send) -> None:\n        scope_type = scope["type"]\n        if scope_type not in ("http", "websocket"):\n            await self.app(scope, receive, send)\n            return\n\n        if scope_type == "http" and scope.get("method") == "OPTIONS":\n            resp = Response(b"", status_code=HTTPStatus.NO_CONTENT)\n            await resp(scope, receive, _with_cors_send(send))\n            return\n\n        send_out = _with_cors_send(send) if scope_type == "http" else send\n        root_path = scope.get("root_path", "")\n        path = scope["path"].removeprefix(root_path)\n        headers = Headers(scope=scope)\n\n        if not _verify_auth(headers):\n            if scope_type == "websocket":\n                await send({"type": "websocket.close", "code": 4401})\n                return\n            resp = JSONResponse(\n                {\n                    "error": {\n                        "message": "Unauthorized: invalid or missing Bearer token",\n                        "type": "AuthenticationError",\n                        "code": 401,\n                    }\n                },\n                status_code=HTTPStatus.UNAUTHORIZED,\n            )\n            await resp(scope, receive, send_out)\n            return\n\n        if any(path == p or path.startswith(p + "/") for p in BLOCKED_ADMIN_PREFIXES):\n            if scope_type == "websocket":\n                await send({"type": "websocket.close", "code": 4403})\n                return\n            resp = JSONResponse(\n                {\n                    "error": {\n                        "message": f"Endpoint {path} is disabled on public inference runtime",\n                        "type": "PermissionError",\n                        "code": 403,\n                    }\n                },\n                status_code=HTTPStatus.FORBIDDEN,\n            )\n            await resp(scope, receive, send_out)\n            return\n\n        if scope_type == "http":\n            method = scope.get("method", "GET")\n\n            if method == "GET" and path == "/api/ct/info":\n                resp = JSONResponse({\n                    "service": "drama3",\n                    "engine": "sglang-omni",\n                    "model": SERVED_MODEL,\n                    "alias": MODEL_ALIAS,\n                    "sample_rate": SAMPLE_RATE,\n                    "max_codec_frames": 2048,\n                    "max_chunk_seconds": 95.1,\n                    "max_reference_seconds": MAX_REF_SEC,\n                    "speed_range": [0.25, 4.0],\n                    "formats": ["wav", "mp3", "flac", "pcm", "aac", "opus"],\n                    "supports_batch": True,\n                    "supports_multi_reference": True,\n                    "speaker_tags": ["<|speaker:0|>"],\n                    "default_ref_wav": str(DEFAULT_REF_WAV),\n                    "drive_voices_dir": str(DRIVE_VOICES_DIR),\n                    "drive_output_dir": str(DRIVE_OUTPUT_DIR),\n                })\n                await resp(scope, receive, send_out)\n                return\n\n            if method == "GET" and path == "/api/ct/logs":\n                qs = parse_qs((scope.get("query_string") or b"").decode("utf-8", errors="replace"))\n                try:\n                    n_lines = int((qs.get("lines") or ["120"])[0])\n                except ValueError:\n                    n_lines = 120\n                resp = JSONResponse(tail_service_log(lines=n_lines))\n                await resp(scope, receive, send_out)\n                return\n\n            if method == "GET" and path == "/api/ct/drive-list":\n                qs = parse_qs((scope.get("query_string") or b"").decode("utf-8", errors="replace"))\n                spec = (qs.get("path") or [""])[0]\n                recursive = (qs.get("recursive") or ["1"])[0] not in ("0", "false", "False")\n                try:\n                    files = await asyncio.to_thread(\n                        list_drive_audio_files,\n                        spec,\n                        data_dir=DATA_DIR,\n                        default_subdir="voices",\n                        recursive=recursive,\n                        mydrive_root=MYDRIVE_ROOT,\n                        drive_root=DRIVE_ROOT,\n                    )\n                    resp = JSONResponse({\n                        "drive_mounted": MYDRIVE_ROOT.is_dir(),\n                        "query": spec,\n                        "count": len(files),\n                        "files": files,\n                    })\n                except Exception as err:\n                    resp = JSONResponse({"error": str(err)}, status_code=400)\n                await resp(scope, receive, send_out)\n                return\n\n            if method == "POST" and path == "/api/ct/drive-upload-voice":\n                chunks: list[bytes] = []\n                while True:\n                    msg = await receive()\n                    if msg["type"] == "http.request":\n                        chunks.append(msg.get("body", b""))\n                        if not msg.get("more_body", False):\n                            break\n                    elif msg["type"] == "http.disconnect":\n                        return\n                try:\n                    body = json.loads(b"".join(chunks))\n                    if not isinstance(body, dict):\n                        raise ValueError("Expected JSON object")\n                    code, out = await asyncio.to_thread(_upload_drive_voice_sync, body)\n                    resp = JSONResponse(out, status_code=code)\n                except ValueError as err:\n                    resp = JSONResponse({"error": str(err)}, status_code=400)\n                except Exception as err:\n                    resp = JSONResponse({"error": str(err)}, status_code=500)\n                await resp(scope, receive, send_out)\n                return\n\n            if method == "POST" and path in ("/v1/audio/speech", "/v1/audio/speech/batch"):\n                is_batch = path.endswith("/batch")\n                chunks = []\n                while True:\n                    msg = await receive()\n                    if msg["type"] == "http.request":\n                        chunks.append(msg.get("body", b""))\n                        if not msg.get("more_body", False):\n                            break\n                    elif msg["type"] == "http.disconnect":\n                        return\n                raw_body = b"".join(chunks)\n                try:\n                    rewritten_body, save_drive_spec, resp_fmt = await asyncio.to_thread(\n                        _prepare_speech_request, raw_body, is_batch=is_batch\n                    )\n                    header_save = headers.get("x-save-drive")\n                    if header_save and not save_drive_spec and not is_batch:\n                        save_drive_spec = await asyncio.to_thread(_validate_save_drive_spec, header_save)\n                except Exception as err:\n                    resp = JSONResponse(\n                        {"error": {"message": str(err), "type": "BadRequestError", "code": 400}},\n                        status_code=400,\n                    )\n                    await resp(scope, receive, send_out)\n                    return\n\n                # Update content-length header for downstream ASGI parser\n                new_headers = [\n                    (k, v) for (k, v) in scope.get("headers", []) if k.lower() != b"content-length"\n                ]\n                new_headers.append((b"content-length", str(len(rewritten_body)).encode("ascii")))\n                patched_scope = {**scope, "headers": new_headers}\n\n                body_sent = False\n\n                async def replay_receive() -> Message:\n                    nonlocal body_sent\n                    if not body_sent:\n                        body_sent = True\n                        return {"type": "http.request", "body": rewritten_body, "more_body": False}\n                    return await receive()\n\n                if not save_drive_spec or is_batch:\n                    await self.app(patched_scope, replay_receive, send_out)\n                    return\n\n                resp_status = 200\n                resp_headers: list[tuple[bytes, bytes]] = []\n                resp_chunks: list[bytes] = []\n                start_msg: Message | None = None\n\n                async def tee_send(message: Message) -> None:\n                    nonlocal resp_status, resp_headers, start_msg\n                    if message["type"] == "http.response.start":\n                        resp_status = message.get("status", 200)\n                        resp_headers = list(message.get("headers", []))\n                        start_msg = message\n                        if resp_status != 200:\n                            await send_out(message)\n                    elif message["type"] == "http.response.body":\n                        chunk = message.get("body", b"")\n                        more = message.get("more_body", False)\n                        if resp_status != 200:\n                            await send_out(message)\n                            return\n                        if chunk:\n                            resp_chunks.append(chunk)\n                        if not more and start_msg is not None:\n                            full_audio = b"".join(resp_chunks)\n                            saved_path = await asyncio.to_thread(\n                                save_output_audio_to_drive,\n                                full_audio,\n                                save_drive_spec,\n                                data_dir=DATA_DIR,\n                                output_dir=DRIVE_OUTPUT_DIR,\n                                prefix="drama3",\n                                fmt=resp_fmt,\n                                mydrive_root=MYDRIVE_ROOT,\n                                drive_root=DRIVE_ROOT,\n                            )\n                            safe_hdr = sanitize_header_param(saved_path).encode("utf-8")\n                            out_headers = [\n                                (k, v) for (k, v) in resp_headers if k.lower() != b"x-saved-drive-path"\n                            ]\n                            out_headers.append((b"x-saved-drive-path", safe_hdr))\n                            await send_out({**start_msg, "headers": out_headers})\n                            await send_out({"type": "http.response.body", "body": full_audio, "more_body": False})\n                    else:\n                        await send_out(message)\n\n                await self.app(patched_scope, replay_receive, tee_send)\n                return\n\n            if method in ("POST", "DELETE") and path.startswith("/v1/audio/voices"):\n                deleted_name: str | None = None\n                if method == "DELETE" and path.startswith("/v1/audio/voices/"):\n                    deleted_name = unquote(path.removeprefix("/v1/audio/voices/").strip("/"))\n                await self.app(scope, receive, send_out)\n                await asyncio.to_thread(_sync_speakers_to_drive, deleted_name)\n                return\n\n        await self.app(scope, receive, send_out)\n\n\ndef run_sgl_omni_server(argv: list[str] | None = None) -> None:\n    """Wrap sglang-omni\'s FastAPI app with Drama3Middleware and launch sgl-omni serve."""\n    import sglang_omni.cli as sgl_cli\n    import sglang_omni.serve.launcher as launcher\n\n    orig_server_init = launcher.PipelineUvicornServer.__init__\n\n    def _wrapped_server_init(self: Any, config: Any, *args: Any, **kwargs: Any) -> None:\n        if not isinstance(config.app, Drama3Middleware):\n            config.app = Drama3Middleware(config.app)\n            if getattr(config, "loaded_app", None) is not None:\n                config.loaded_app = config.app\n        orig_server_init(self, config, *args, **kwargs)\n\n    launcher.PipelineUvicornServer.__init__ = _wrapped_server_init  # type: ignore[method-assign]\n\n    orig_async_client = launcher.httpx.AsyncClient\n\n    def _authed_async_client(*args: Any, **kwargs: Any) -> Any:\n        hdrs = dict(kwargs.get("headers") or {})\n        if API_TOKEN:\n            hdrs.setdefault("Authorization", f"Bearer {API_TOKEN}")\n        kwargs["headers"] = hdrs\n        return orig_async_client(*args, **kwargs)\n\n    launcher.httpx.AsyncClient = _authed_async_client  # type: ignore[assignment]\n\n    if argv is not None:\n        sys.argv = [sys.argv[0], *argv]\n    sgl_cli.app()\n\n\nif __name__ == "__main__":\n    run_sgl_omni_server()\n', encoding="utf-8")

SGL_ENV = {
    "PATH": f"{VENV_DIR}/bin:/usr/local/cuda/bin:{os.environ.get('PATH', '')}",
    "PYTHONPATH": f"{D3_WORK}:{os.environ.get('PYTHONPATH', '')}",
    "VIRTUAL_ENV": str(VENV_DIR),
    "FLASHINFER_DISABLE_VERSION_CHECK": "1",
    "FLASHINFER_WORKSPACE_BASE": str(FLASHINFER_CACHE_DIR),
    "TRITON_CACHE_DIR": str(TRITON_CACHE_DIR),
    "TORCHINDUCTOR_CACHE_DIR": str(INDUCTOR_CACHE_DIR),
    "SPEAKER_SAMPLES_DIR": str(SPEAKERS_DIR),
    "CT_API_TOKEN": API_TOKEN,
    "CT_SERVICE_PORT": str(SERVICE_PORT),
    "CT_DATA_DIR": str(DATA_DIR),
    "CT_DRAMA3_MODEL": MODEL_REPO,
    "CT_DRAMA3_ALIAS": MODEL_ALIAS,
    "CT_DRAMA3_MAX_REF_SEC": str(MAX_REF_SEC),
    "CT_DEFAULT_REF_WAV": str(DEFAULT_REF_WAV),
    "CT_DEFAULT_REF_TEXT": DEFAULT_REF_TEXT,
}

SERVER_ARGS = [
    str(VENV_DIR / "bin" / "python"),
    str(D3_WORK / "ct_drama3_middleware.py"),
    "serve",
    "--model-path", str(MODEL_DIR),
    "--host", "127.0.0.1",
    "--port", str(SERVICE_PORT),
    "--model-name", MODEL_REPO,
    "--allowed-local-media-path", str(D3_WORK),
    "--tts-batch-max-items", str(BATCH_MAX_ITEMS),
    "--tts_engine.gpu_memory_fraction", str(ENGINE_GPU_MEM),
    "--vocoder.gpu_memory_fraction", str(VOCODER_GPU_MEM),
    "--vocoder.factory.codec_path", str(CODEC_DIR),
    "--tts_engine.factory.max_running_requests", str(MAX_RUNNING_REQS),
    "--tts_engine.factory.context_length", str(CONTEXT_LENGTH),
    "--preprocessing.factory.max_reference_audio_seconds", str(MAX_REF_SEC),
]

def start_service():
    subprocess.run(["pkill", "-9", "-f", "ct_drama3_middleware|sglang_omni"], check=False)
    time.sleep(1)
    dlog(f"Starting SGLang-Omni Drama 3 server on 127.0.0.1:{SERVICE_PORT} (model={MODEL_REPO})")
    return ct_spawn("service", SERVER_ARGS, env=SGL_ENV, cwd=str(D3_WORK))

PROCS["service"] = (start_service(), start_service)


In [ ]:
# @title 3. Auth layer -- Caddy: basic auth for browsers, `Authorization: Bearer <token>` for API clients
# PUBLIC_PORT must be free on the VM: Colab itself uses 8080 (runtime server) and 8012 (colab dfs shim).
# Only used when the service has no auth of its own (CT_AUTH=caddy). The service itself listens on
# 127.0.0.1 only, cloudflared targets Caddy, so every public request passes through auth.
REWRITE_HOST = True   # True: send Host: 127.0.0.1:<port> upstream (Ollama & co. reject foreign Host headers)
import platform
CADDY_ARCH = {"x86_64": "amd64", "aarch64": "arm64"}.get(platform.machine(), "amd64")

def start_caddy():
    if not os.path.exists("/usr/local/bin/caddy"):
        tag = json.loads(urllib.request.urlopen(
            "https://api.github.com/repos/caddyserver/caddy/releases/latest", timeout=30).read())["tag_name"]
        tgz = "/tmp/caddy.tgz"
        with urllib.request.urlopen(f"https://github.com/caddyserver/caddy/releases/download/{tag}/"
                                    f"caddy_{tag.lstrip('v')}_linux_{CADDY_ARCH}.tar.gz", timeout=60) as src, \
                open(tgz, "wb") as dst:
            shutil.copyfileobj(src, dst)
        subprocess.run(["tar", "-xzf", tgz, "-C", "/usr/local/bin", "caddy"], check=True)
    import hashlib
    cookie_tok = hashlib.sha256(f"ct-cookie-v2:{AUTH_USER}:{AUTH_PASS}:{API_TOKEN}".encode()).hexdigest()
    hashed = subprocess.check_output(["caddy", "hash-password", "--plaintext", AUTH_PASS], text=True).strip()
    upstream = f"""reverse_proxy 127.0.0.1:{SERVICE_PORT} {{
            {"header_up Host {upstream_hostport}" if REWRITE_HOST else ""}
            {"header_up Origin http://{upstream_hostport}" if REWRITE_HOST else ""}
            header_up -Authorization
            flush_interval -1
        }}"""
    caddyfile = f"""{{
    admin off
    auto_https off
}}
:{PUBLIC_PORT} {{
    encode zstd gzip
    @bearer header Authorization "Bearer {API_TOKEN}"
    handle @bearer {{
        {upstream}
    }}
    @cookie header Cookie *ct_session={cookie_tok}*
    handle @cookie {{
        {upstream}
    }}
    handle {{
        route {{
            basic_auth {{
                "{AUTH_USER}" {hashed}
            }}
            header +Set-Cookie "ct_session={cookie_tok}; Path=/; HttpOnly; SameSite=Lax"
            {upstream}
        }}
    }}
}}
"""
    open(f"{WORK}/Caddyfile", "w").write(caddyfile)
    subprocess.run(["pkill", "-f", "caddy run"], check=False)
    return ct_spawn("caddy", ["caddy", "run", "--config", f"{WORK}/Caddyfile", "--adapter", "caddyfile"])

AUTH_OK = False   # cell 4 refuses to open the tunnel unless this becomes True (fail closed)
if AUTH == "caddy":
    PROCS["caddy"] = (start_caddy(), start_caddy)
    time.sleep(2)
    if PROCS["caddy"][0].poll() is not None:   # e.g. port taken (Colab uses 8080 itself)
        raise RuntimeError(f"caddy exited -- is :{PUBLIC_PORT} in use?\n" + ct_tail("caddy"))
    # Must be 401 from *our* Caddy and must NOT leak Set-Cookie on unauthenticated 401 responses.
    ct_wait_http(f"http://127.0.0.1:{PUBLIC_PORT}/", timeout=60, ok=lambda c: c == 401)
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PUBLIC_PORT}/", timeout=5)
    except urllib.error.HTTPError as _e:
        if _e.headers.get("Set-Cookie"):
            raise RuntimeError("Caddy leaked Set-Cookie on unauthenticated 401 response!")
    AUTH_OK = True
    print(f"caddy on :{PUBLIC_PORT} -> 127.0.0.1:{SERVICE_PORT} (unauthenticated request -> 401)")
else:
    AUTH_OK = True
    print(f"auth={AUTH}: no proxy, cloudflared targets the service on :{PUBLIC_PORT} directly")


In [ ]:
# @title 4. Tunnel -- Cloudflare by default, or project-configured FRP + HTTPS
"""Self-contained Colab transport cell; injected into every service notebook."""
import hashlib
import json
import os
import platform
import re
import shutil
import signal
import subprocess
import tarfile
import tempfile
import threading
import time
import urllib.error
import urllib.request
from pathlib import Path


def render_frpc(config, local_port, ca_path):
    """TOML strings use JSON quoting, never shell interpolation."""
    return f'''serverAddr = {json.dumps(config['server_ip'])}
serverPort = {int(config['server_port'])}
loginFailExit = false
auth.method = "token"
auth.token = {json.dumps(config['token'])}
auth.additionalScopes = {json.dumps(config.get('auth_scopes',['HeartBeats','NewWorkConns']))}
transport.tls.enable = true
transport.tls.trustedCaFile = {json.dumps(str(ca_path))}
transport.tls.serverName = {json.dumps(config.get('server_name') or config['server_ip'])}
log.to = "console"
log.level = "info"
[[proxies]]
name = {json.dumps(config['proxy_name'])}
type = "tcp"
localIP = "127.0.0.1"
localPort = {int(local_port)}
remotePort = {int(config['remote_port'])}
'''


def install_frpc(destination, version):
    if destination.exists():
        return str(destination)
    arch={'x86_64':'amd64','aarch64':'arm64'}.get(platform.machine())
    if not arch:
        raise ValueError('Unsupported FRP client architecture')
    if not re.fullmatch(r'\d+\.\d+\.\d+',version) or tuple(map(int,version.split('.')))<(0,61,0):
        raise ValueError('Unsupported FRP client version; expected stable >= 0.61.0')
    asset=f'frp_{version}_linux_{arch}.tar.gz'
    base=f'https://github.com/fatedier/frp/releases/download/v{version}/'
    with tempfile.TemporaryDirectory() as tmp:
        archive=Path(tmp)/asset
        with urllib.request.urlopen(base+asset,timeout=60) as response, open(archive,'wb') as out:
            shutil.copyfileobj(response,out)
        checks=urllib.request.urlopen(base+'frp_sha256_checksums.txt',timeout=60).read().decode()
        expected=next((line.split()[0] for line in checks.splitlines()
                       if line.split() and line.split()[-1].lstrip('*')==asset),None)
        if expected is None or hashlib.sha256(archive.read_bytes()).hexdigest()!=expected:
            raise ValueError('FRP client checksum verification failed')
        with tarfile.open(archive) as tar:
            entry=tar.getmember(f'frp_{version}_linux_{arch}/frpc')
            if not entry.isfile():
                raise ValueError('Invalid FRP client binary')
            destination.parent.mkdir(parents=True,exist_ok=True)
            with tar.extractfile(entry) as source:
                destination.write_bytes(source.read())
            destination.chmod(0o755)
    return str(destination)


def run_tunnel(ctx):
    if not ctx.get('AUTH_OK'):
        raise RuntimeError('Auth layer not ready; refusing to expose the service')
    work=Path(ctx['WORK'])
    transport=os.environ.get('CT_TRANSPORT','cloudflare')
    if transport not in ('cloudflare','frp'):
        raise ValueError('Unsupported tunnel transport')
    # Stop only the tunnel previously started by this notebook (PID + proc
    # start time prevent accidental termination after PID reuse).
    marker=work/'tunnel-process.json'
    def identity(pid):
        try:
            stat=Path(f'/proc/{pid}/stat').read_text().rsplit(')',1)[1].split()
            return stat[19]
        except OSError:
            return None
    if marker.exists():
        previous=json.loads(marker.read_text())
        if previous.get('start_time') and identity(previous['pid'])==previous['start_time']:
            os.kill(previous['pid'],signal.SIGTERM)
            for _ in range(50):
                if identity(previous['pid']) is None:
                    break
                time.sleep(.1)
    def spawn(name,args):
        (work/(name+'.log')).write_text('')
        proc=ctx['ct_spawn'](name,args)
        marker.write_text(json.dumps(dict(pid=proc.pid,start_time=identity(proc.pid))))
        return proc

    if transport=='frp':
        config={key:os.environ['CT_FRP_'+key.upper()] for key in
                ('server_ip','server_port','remote_port','proxy_name','token')}
        config['server_name']=os.environ.get('CT_FRP_SERVER_NAME') or config['server_ip']
        config['auth_scopes']=json.loads(os.environ.get('CT_FRP_AUTH_SCOPES') or '["HeartBeats","NewWorkConns"]')
        public_host=ctx['PUBLIC_HOST']
        if not public_host:
            raise ValueError('FRP requires a public hostname')
        ca_path=work/'frp-ca.crt'
        ca_path.write_text(os.environ['CT_FRP_CA_CERT'])
        config_path=work/'frpc.toml'
        config_path.touch(mode=0o600,exist_ok=True)
        config_path.chmod(0o600)
        config_path.write_text(render_frpc(config,ctx['PUBLIC_PORT'],ca_path))
        binary=install_frpc(work/('frpc-'+os.environ['CT_FRP_VERSION']),os.environ['CT_FRP_VERSION'])
        result=subprocess.run([binary,'verify','-c',str(config_path)],capture_output=True)
        if result.returncode:
            raise RuntimeError('FRP client configuration validation failed')
        def start():
            return spawn('frpc',[binary,'-c',str(config_path)])
        key='frpc'
    else:
        cf_arch={'x86_64':'amd64','aarch64':'arm64'}.get(platform.machine(),'amd64')
        binary=ctx['ct_fetch_binary']('/usr/local/bin/cloudflared',
            f'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-{cf_arch}')
        def refresh_quick_url(proc_ref, wait_seconds=60):
            deadline_q=time.monotonic()+wait_seconds
            while time.monotonic()<deadline_q:
                if proc_ref.poll() is not None:
                    return None
                match_q=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',ctx['ct_tail']('cloudflared',100))
                if match_q:
                    new_url=match_q.group(0)
                    if new_url!=ctx.get('PUBLIC_URL'):
                        ctx['PUBLIC_URL']=new_url
                        print(f'CT_PUBLIC_URL={new_url}',flush=True)
                    return new_url
                time.sleep(1)
            return None
        ctx['ct_refresh_public_url']=refresh_quick_url
        def start():
            args=[binary,'tunnel','--no-autoupdate','--protocol','http2']
            if ctx['TUNNEL_TOKEN']:
                args+=['run','--token',ctx['TUNNEL_TOKEN']]
            else:
                args+=['--url',f'http://127.0.0.1:{ctx["PUBLIC_PORT"]}']
            proc_new=spawn('cloudflared',args)
            if not ctx['TUNNEL_TOKEN'] and ctx.get('PUBLIC_URL'):
                threading.Thread(target=refresh_quick_url,args=(proc_new,),daemon=True).start()
            return proc_new
        key='cloudflared'
    proc=start()
    ctx['PROCS'][key]=(proc,start)
    ctx['TUNNEL_PROCESS_KEY']=key
    deadline=time.monotonic()+180
    registered=False
    while time.monotonic()<deadline:
        if proc.poll() is not None:
            raise RuntimeError(f'{key} exited before registration; see /content/ct/{key}.log')
        log=ctx['ct_tail'](key,100)
        if transport=='frp':
            registered=registered or ('start proxy success' in log and config['proxy_name'] in log)
            if registered:
                url='https://'+ctx['PUBLIC_HOST']
                try:
                    with urllib.request.urlopen(url,timeout=5) as response:
                        if response.status<600:
                            break
                except urllib.error.HTTPError as error:
                    # A valid public TLS response proves the transport/site.
                    # 502 is expected while a large model is still loading;
                    # the service's keep-alive cell gates CT_READY on health.
                    code=error.code
                    error.close()
                    if code<600:
                        break
                except OSError:
                    pass
        elif ctx['TUNNEL_TOKEN'] and 'Registered tunnel connection' in log:
            url='https://'+ctx['PUBLIC_HOST']
            break
        elif not ctx['TUNNEL_TOKEN']:
            match=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',log)
            if match:
                url=match.group(0)
                break
        time.sleep(1)
    else:
        raise RuntimeError(f'{key} not ready; check tunnel logs, DNS, HTTPS and server ports')
    ctx['PUBLIC_URL']=url
    print(f'CT_PUBLIC_URL={url}',flush=True)

run_tunnel(globals())


In [ ]:
# @title 5. Ready + keep-alive -- wait for CUDA graph capture, verify auth, register default voice, warmup & sync caches
BROWSER_UA = "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/132.0.0.0 Safari/537.36"

def http_json(url, key=None, payload=None, timeout=30):
    headers = {"Content-Type": "application/json", "User-Agent": BROWSER_UA}
    if key is not None:
        headers["Authorization"] = "Bearer " + key
    req = urllib.request.Request(
        url, headers=headers, data=None if payload is None else json.dumps(payload).encode("utf-8")
    )
    try:
        with urllib.request.urlopen(req, timeout=timeout) as resp:
            raw = resp.read().decode("utf-8", errors="replace")
            return resp.status, (json.loads(raw) if raw.strip().startswith(("{", "[")) else {})
    except urllib.error.HTTPError as err:
        return err.code, {}

def auth_check(base, key):
    for candidate, expected in ((None, 401), ("invalid-token-probe", 401), (key, 200)):
        code, _ = http_json(base + "/v1/models", candidate)
        if code != expected:
            raise RuntimeError(f"Auth test failed on {base}/v1/models: expected {expected}, got {code}")

LOCAL_BASE = f"http://127.0.0.1:{SERVICE_PORT}"
deadline = time.monotonic() + 2400
last_notice = 0
while time.monotonic() < deadline:
    proc = PROCS["service"][0]
    if proc.poll() is not None:
        raise RuntimeError("SGLang-Omni Drama 3 exited during startup:\n" + ct_tail("service", 90))
    try:
        code, _ = http_json(LOCAL_BASE + "/health", API_TOKEN, timeout=10)
        if code == 200:
            break
    except Exception:
        pass
    if time.monotonic() - last_notice >= 25:
        gpu_stat = ct_gpu_stat()
        last_line = (ct_tail("service", 3).strip().splitlines() or ["starting..."])[-1]
        dlog(f"Waiting for Drama 3 pipeline & CUDA graphs | GPU: {gpu_stat} | {last_line[:100]}")
        last_notice = time.monotonic()
    time.sleep(5)
else:
    raise RuntimeError("Timed out waiting for SGLang-Omni Drama 3 server to become ready:\n" + ct_tail("service", 90))

auth_check(LOCAL_BASE, API_TOKEN)

# Register default 'alloy' voice preset if not yet saved
code, voices_data = http_json(LOCAL_BASE + "/v1/audio/voices", API_TOKEN, timeout=20)
existing_voices = set(voices_data.get("voices") or [])
for preset_voice in ("alloy",):
    if preset_voice not in existing_voices and DEFAULT_REF_WAV.is_file():
        v_code, _ = http_json(
            LOCAL_BASE + "/api/ct/drive-upload-voice",
            API_TOKEN,
            payload={
                "name": preset_voice,
                "use_default_ref": True,
                "consent": "builtin-default-reference",
                "speaker_description": "Default English reference voice for Fish Audio Drama 3",
            },
            timeout=90,
        )
        dlog(f"Registered preset voice '{preset_voice}': HTTP {v_code}")

# Warmup synthesis to populate Triton/FlashInfer JIT kernels before saving cache archives
warmup_req = urllib.request.Request(
    LOCAL_BASE + "/v1/audio/speech",
    headers={"Content-Type": "application/json", "Authorization": f"Bearer {API_TOKEN}"},
    data=json.dumps({
        "model": MODEL_REPO,
        "input": "Hello, the Fish Audio Drama 3 speech service is ready.",
        "response_format": "wav",
        "max_new_tokens": 384,
    }, ensure_ascii=False).encode("utf-8"),
)
try:
    with urllib.request.urlopen(warmup_req, timeout=180) as wresp:
        wav_bytes = wresp.read()
        dlog(f"Warmup synthesis succeeded ({len(wav_bytes):,} bytes 44.1 kHz WAV)")
except Exception as werr:
    dlog(f"Warmup synthesis warning: {werr}")

for _local_c, _legacy_c in CACHE_PAIRS:
    try:
        save_cache(_local_c, CACHE_ARCHIVES, CACHE_KEY, dlog)
    except (OSError, ValueError) as err:
        dlog(f"Cache archive save warning ({_local_c.name}): {err}")
subprocess.run(["cp", "-ru", f"{SPEAKERS_DIR}/.", f"{DRIVE_VOICES_DIR}/"], check=False)

ct_public_gate(
    PUBLIC_URL,
    lambda: http_json(PUBLIC_URL + "/v1/models", API_TOKEN, timeout=15)[0],
    lambda: auth_check(PUBLIC_URL, API_TOKEN),
    dlog,
    timeout=180,
    interval=8,
)
dlog(f"Public endpoint verified: {PUBLIC_URL}/v1/models")

print(f"{SERVICE} is up: {PUBLIC_URL} (model={MODEL_REPO})")
print("CT_READY", flush=True)

ct_watchdog(
    lambda: http_json(LOCAL_BASE + "/health", API_TOKEN, timeout=10)[0] == 200,
    start_service,
    dlog,
    heartbeat=lambda up: dlog(f"heartbeat | up={up}s | GPU: {ct_gpu_stat()}"),
)
